# Взлом шифра Виженера
## Задача: расшифровать cipher.txt

In [ ]:
def index_of_coincidence(text: str) -> float:
    """Вычисляет индекс совпадений для текста"""
    text = ''.join([c for c in text if c.isalpha()]).lower()
    if len(text) < 2:
        return 0
    n = len(text)
    from collections import Counter
    freq = Counter(text)
    ic = sum(count * (count - 1) for count in freq.values()) / (n * (n - 1))
    return ic


def find_key_length(ciphertext: str, max_len: int = 30) -> int:
    """Находит длину ключа методом индекса совпадений"""
    best_len = 1
    best_ic_diff = float('inf')
    
    for key_len in range(1, max_len + 1):
        avg_ic = 0
        for i in range(key_len):
            segment = ciphertext[i::key_len]
            avg_ic += index_of_coincidence(segment)
        avg_ic /= key_len
        # Для русского языка IC ~ 0.055-0.06
        diff = abs(avg_ic - 0.055)
        if diff < best_ic_diff:
            best_ic_diff = diff
            best_len = key_len
    
    return best_len


def find_key(ciphertext: str, key_len: int) -> str:
    """Находит ключ заданной длины методом частотного анализа"""
    key = ""
    # Частоты букв русского языка
    russian_freq = {
        'о': 0.1098, 'е': 0.0845, 'а': 0.0801, 'и': 0.0735, 'н': 0.0670,
        'т': 0.0626, 'с': 0.0547, 'р': 0.0473, 'в': 0.0454, 'л': 0.0440,
        'к': 0.0349, 'м': 0.0321, 'д': 0.0298, 'п': 0.0280, 'у': 0.0262,
        'я': 0.0201, 'ы': 0.0190, 'ь': 0.0174, 'г': 0.0170, 'з': 0.0165,
        'б': 0.0159, 'ч': 0.0144, 'й': 0.0121, 'х': 0.0097, 'ж': 0.0094,
        'ш': 0.0073, 'ю': 0.0064, 'ц': 0.0048, 'щ': 0.0036, 'э': 0.0032,
        'ф': 0.0026, 'ъ': 0.0004, 'ё': 0.0004
    }
    
    alphabet = 'абвгдежзийклмнопрстуфхцчшщъыьэюя'
    
    for i in range(key_len):
        segment = ciphertext[i::key_len]
        segment = ''.join([c for c in segment if c.isalpha()]).lower()
        best_shift = 0
        best_score = 0
        
        for shift in range(26):
            # Расшифровываем сегмент с данным сдвигом
            decrypted = ""
            for char in segment:
                if char in alphabet:
                    idx = alphabet.index(char)
                    decrypted += alphabet[(idx - shift) % len(alphabet)]
            
            # Оценка по частотам
            score = 0
            from collections import Counter
            freq = Counter(decrypted)
            total = len(decrypted)
            for letter, expected_freq in russian_freq.items():
                actual_freq = freq.get(letter, 0) / total if total > 0 else 0
                score += expected_freq * actual_freq
            
            if score > best_score:
                best_score = score
                best_shift = shift
        
        key += alphabet[best_shift % len(alphabet)]
    
    return key


# Загружаем зашифрованный текст
with open('cipher.txt', 'r', encoding='utf-8') as f:
    ciphertext = f.read()

print("Зашифрованный текст (первые 200 символов):")
print(ciphertext[:200])
print("\n" + "="*50 + "\n")

# Находим длину ключа
key_len = find_key_length(ciphertext, max_len=20)
print(f"Предполагаемая длина ключа: {key_len}")

# Находим ключ
key = find_key(ciphertext, key_len)
print(f"Найденный ключ: {key}")

# Расшифровываем
def decrypt_vigenere(ciphertext, key):
    alphabet = 'абвгдежзийклмнопрстуфхцчшщъыьэюя'
    key = key.lower()
    result = []
    key_index = 0
    
    for char in ciphertext:
        if char.isalpha():
            shift = alphabet.index(key[key_index % len(key)])
            if char in alphabet:
                idx = alphabet.index(char)
                result.append(alphabet[(idx - shift) % len(alphabet)])
            elif char.lower() in alphabet:
                # Русская буква в другом регистре
                idx = alphabet.index(char.lower())
                dec = alphabet[(idx - shift) % len(alphabet)]
                result.append(dec.upper() if char.isupper() else dec)
            else:
                result.append(char)
            key_index += 1
        else:
            result.append(char)
    
    return ''.join(result)

plaintext = decrypt_vigenere(ciphertext, key)

print("\n" + "="*50 + "\n")
print("Расшифрованный текст (первые 500 символов):")
print(plaintext[:500])

# Сохраняем результат
with open('cracked.txt', 'w', encoding='utf-8') as f:
    f.write(plaintext)

print("\nРезультат сохранён в cracked.txt")